# Colab Bootstrap — GNN-NIDS

Chạy notebook này đầu tiên trong mỗi phiên Colab. Nó sẽ:
1. Mount Google Drive (nơi chứa dữ liệu nặng + checkpoint + MLflow log).
2. Clone (lần đầu) hoặc pull (các lần sau) code từ GitHub repo.
3. Cài các thư viện trong `requirements.txt`.
4. Thêm `src/` vào `sys.path` để import module viết trong VS Code.

**Trước khi chạy:** điền `GITHUB_USER`, `GITHUB_REPO` ở cell bên dưới.

Nếu repo private: vào Colab menu trái > icon chìa khoá (Secrets) > thêm secret tên `GITHUB_TOKEN` = Personal Access Token (scope `repo`), bật "Notebook access". Không paste token trực tiếp vào cell.

In [ ]:
GITHUB_USER = "thunww"
GITHUB_REPO = "gnn-nids-tttn"

DRIVE_ROOT = "/content/drive/MyDrive/GNN-NIDS_TTTN"      # nơi lưu data/checkpoint/mlflow trên Drive
PROJECT_DIR = "/content/GNN-NIDS_TTTN"                    # nơi clone code từ GitHub về

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.makedirs(f"{DRIVE_ROOT}/data", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/mlruns", exist_ok=True)

In [ ]:
import os

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None  # repo public: không cần token

auth = f"{token}@" if token else ""
remote_url = f"https://{auth}github.com/{GITHUB_USER}/{GITHUB_REPO}.git"

if os.path.isdir(PROJECT_DIR):
    print("Repo đã tồn tại, pull bản mới nhất...")
    !cd {PROJECT_DIR} && git pull
else:
    print("Clone repo lần đầu...")
    !git clone {remote_url} {PROJECT_DIR}

In [ ]:
!pip install -q -r {PROJECT_DIR}/requirements.txt

In [ ]:
import sys
sys.path.append(f"{PROJECT_DIR}/src")

# Từ đây import module viết trong VS Code, vd:
# from etl.load import load_netflow_dataset

print("Sẵn sàng. PROJECT_DIR =", PROJECT_DIR, "| DRIVE_ROOT =", DRIVE_ROOT)

## Giải nén dữ liệu raw từ Drive

Chạy 1 lần đầu tiên (hoặc khi thêm dataset mới). Yêu cầu: đã upload các file `.zip` dữ liệu vào `MyDrive/GNN-NIDS_TTTN/data/` trước (tên file phải khớp với key trong dict `zips` bên dưới).

In [ ]:
import zipfile, os, shutil

zips = {
    "nf-cse-cic-ids2018-v2.zip": "nf-cse-cic-ids2018-v2",
    "nf-unsw-nb15-v2.zip": "nf-unsw-nb15-v2",
}

for zip_name, folder_name in zips.items():
    zip_path = f"{DRIVE_ROOT}/data/{zip_name}"
    out_dir = f"{DRIVE_ROOT}/data/raw/{folder_name}"

    if os.path.isdir(out_dir) and os.listdir(out_dir):
        print("Bỏ qua (đã có dữ liệu):", folder_name, "->", os.listdir(out_dir))
        continue

    os.makedirs(out_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(out_dir)

    # zip có thể chứa sẵn 1 thư mục con cùng tên -> đẩy phẳng ra ngoài
    nested = os.path.join(out_dir, folder_name)
    if os.path.isdir(nested):
        for fname in os.listdir(nested):
            shutil.move(os.path.join(nested, fname), os.path.join(out_dir, fname))
        os.rmdir(nested)

    print("Đã giải nén:", folder_name, "->", os.listdir(out_dir))